# ユニットテストを Colab で走らせる

ローカル PC がメモリ枯渇で落ちるため、テストの実行環境を Colab に逃がす。

上から順に実行するだけでよい。`BRANCH` だけ必要に応じて書き換える。

- 認証は不要（リポジトリは公開）
- `chromadb` は入れない。旧 `udemy3.py` だけが import しており、`testpaths = tests` の対象外
- `-m "not integration"` は `pytest.ini` の既定。実機 OCR と Ollama を要するテストは走らない
- したがって GPU は使わないが、L4 ランタイムのままでも支障はない

In [ ]:
REPO = "https://github.com/Hide369/local-llm-rag.git"
BRANCH = "feat/fold-duplicate-chunks"
WORKDIR = "/content/local-llm-rag"

In [ ]:
# 1. 取得（2回目以降は最新を取り直す）
import os, subprocess, sys

def run(*args, cwd=None):
    print("$", " ".join(args))
    r = subprocess.run(args, cwd=cwd, text=True, capture_output=True)
    print(r.stdout + r.stderr)
    if r.returncode:
        raise SystemExit(f"failed: {' '.join(args)}")

if not os.path.isdir(WORKDIR):
    run("git", "clone", "--branch", BRANCH, REPO, WORKDIR)
else:
    run("git", "fetch", "origin", BRANCH, cwd=WORKDIR)
    run("git", "checkout", BRANCH, cwd=WORKDIR)
    run("git", "reset", "--hard", f"origin/{BRANCH}", cwd=WORKDIR)

run("git", "log", "--oneline", "-3", cwd=WORKDIR)
print(sys.version)

In [ ]:
# 2. 依存の導入。chromadb は requirements から除去済みだが、古いコミットを
#    検証するときのために念のため除外する（通常は何もしない）
req = os.path.join(WORKDIR, "requirements.txt")
trimmed = "/content/requirements-colab.txt"
with open(req, encoding="utf-8") as f:
    lines = [l for l in f if not l.strip().startswith("chromadb")]
with open(trimmed, "w", encoding="utf-8") as f:
    f.writelines(lines)
print("".join(l for l in lines if l.strip() and not l.startswith("#")))

run(sys.executable, "-m", "pip", "install", "-q", "-r", trimmed)

In [ ]:
# 3. テスト実行。pytest.ini を効かせるためリポジトリ直下で走らせる
r = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "--tb=short"],
    cwd=WORKDIR, text=True, capture_output=True,
)
print(r.stdout[-20000:])
print(r.stderr[-4000:])
# 何を検査したのかを結果の直後に出す。貼り付けた末尾だけで
# ブランチの取り違えに気づけるようにするため。
head = subprocess.run(
    ["git", "log", "--oneline", "-1"], cwd=WORKDIR, text=True, capture_output=True
).stdout.strip()
print(f"tested: {BRANCH} @ {head}")
print("exit code:", r.returncode)
assert r.returncode == 0, "テストが失敗しています。上の出力を確認してください。"